# Theorem 24 — window-local Laplace adequacy

**Formal source:** [`../24_window_local_laplace_adequacy.md`](../24_window_local_laplace_adequacy.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
times = np.linspace(0, 1, 1000)
rho, omega = 1.2, 18.0
modal = np.exp(-rho * times) * (np.cos(omega * times) + 0.3 * np.sin(omega * times))
residual = 0.02 * np.sin(37 * times)
true_signal = modal + residual
gain = 3
acquisition_error = np.linalg.norm(gain * true_signal - gain * modal)
operator_bound = gain * np.linalg.norm(residual)
assert acquisition_error <= operator_bound + 1e-12
representation_error = np.linalg.norm(np.tanh(gain * true_signal) - np.tanh(gain * modal))
assert representation_error <= acquisition_error + 1e-12
switch_times = np.linspace(0, 1, 500)
switching = np.where(switch_times < 0.5, np.exp(-switch_times), np.exp(-0.5) * np.exp(-4 * (switch_times - 0.5)))
fit = np.polyfit(switch_times, np.log(switching), 1)
single_pole = np.exp(fit[1] + fit[0] * switch_times)
misspecification = np.linalg.norm(switching - single_pole) / np.linalg.norm(switching)
assert misspecification > 0.1
print({"acquisition_error": float(acquisition_error), "operator_bound": float(operator_bound), "representation_error": float(representation_error), "switching_residual": float(misspecification)})

In [ ]:
print('THEORY_DEMO_PASS::24_window_local_laplace_adequacy')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')